In [1]:
# File I/O Agent with PDF RAG using LangChain + LangGraph
# Features: PDF Ingestion ➜ ChromaDB ➜ LangGraph Agent with Retrieval
import os
from typing import TypedDict
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain_core.runnables import RunnableLambda
from langchain.chains import RetrievalQA

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================
# Load Env & Setup Paths
# ============================
load_dotenv()
PDF_PATH = "./data.pdf"  # your PDF file
CHROMA_PATH = "./chroma_db"

In [3]:
# ============================
# PDF → Text → Embeddings → ChromaDB
# ============================
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
splits = text_splitter.split_documents(documents)

In [4]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model, persist_directory=CHROMA_PATH)
retriever = vectorstore.as_retriever()

C:\Users\emon1\AppData\Local\Temp\ipykernel_4920\2262722917.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [5]:
# ============================
# LLM Setup
# ============================
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0.7)
rag_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [6]:
# ============================
# LangGraph State
# ============================
class PDFState(TypedDict):
    input: str
    output: str

In [7]:
# ============================
# Node Logic
# ============================
def rag_node(state: PDFState) -> PDFState:
    print(f"RAG Querying: {state['input']}")
    answer = rag_chain.invoke({"query": state["input"]})
    state["output"] = answer["result"]
    return state

In [8]:
# ============================
# LangGraph
# ============================
builder = StateGraph(PDFState)
builder.set_entry_point("rag")
builder.add_node("rag", rag_node)
builder.add_edge("rag", END)

graph = builder.compile()

In [9]:
# ============================
# Example Run
# ============================
example = {
    "input": "What is the main idea of this document?",
    "output": ""
}

In [10]:
result = graph.invoke(example)
print("\nFINAL RAG ANSWER:\n", result["output"])

RAG Querying: What is the main idea of this document?

FINAL RAG ANSWER:
 Based on the provided context, the main idea of this document is to explain the concepts of cryptography and security.

The document defines cryptography as the science of transforming messages to make them secure. It is structured like a book, with parts dedicated to:

*   Symmetric-Key Encipherment
*   Asymmetric-Key Encipherment
*   Integrity, Authentication, and Key Management
*   Network Security

It also discusses security goals and related attacks.
